# A panorama, without any widgets

Scripted rendering: turn the camera through a full circle, keep every frame, and lay them
out side by side. This is the shape of most real uses of the renderer — a loop issuing
REST calls — and it needs nothing but the client and Pillow, which `peaknav` already
depends on.

Needs Java, a display and the renderer jar, as notebook 02 does.

In [ ]:
from io import BytesIO

from PIL import Image

from peaknav.headless import PeakNavHeadless

SUMMIT = (45.9763, 7.6586)      # the Matterhorn
STEPS = 8                       # every 45 degrees

In [ ]:
nav = PeakNavHeadless(*SUMMIT, width=640, height=400, image_format="jpg")
nav.move_to(*SUMMIT, await_tiles_ms=120_000)
nav.set_altitude_asl(4600)
nav.set_view(sky=True, sky_mode="day", labels=["peaks"])
nav.wait(tiles_timeout_ms=60_000, settle_ms=500)

One frame per bearing. The wait between frames is what keeps a half-drawn tile out of the
picture; with the camera turning in place rather than travelling, there is little new
terrain to stream, so it is quick after the first.

In [ ]:
import time

try:
    from tqdm.auto import tqdm
except ImportError:                      # tqdm is not a dependency of peaknav
    def tqdm(iterable, **kwargs):
        print(kwargs.get("desc", ""), "- pip install tqdm for a progress bar")
        return iterable

frames = []
started = time.time()
for step in tqdm(range(STEPS), desc="panorama", unit="frame"):
    bearing = step * 360 / STEPS
    nav.look(bearing_deg=bearing, pitch_deg=-4)
    # Bounded, and short: the camera turns in place, so there is little new terrain to
    # stream and a long timeout only buys a long wait when something is wrong. Every step
    # prints, so a slow run looks slow rather than looking hung.
    nav.wait(tiles_timeout_ms=8_000, settle_ms=200)
    frames.append((bearing, Image.open(BytesIO(nav.frame("jpg")))))

## The contact sheet

In [ ]:
def contact_sheet(images, columns=4, pad=6):
    """Lays the frames out in a grid, on one canvas."""
    width, height = images[0].size
    rows = (len(images) + columns - 1) // columns
    sheet = Image.new("RGB",
                      (columns * width + (columns + 1) * pad,
                       rows * height + (rows + 1) * pad),
                      (250, 250, 250))
    for index, image in enumerate(images):
        row, column = divmod(index, columns)
        sheet.paste(image, (pad + column * (width + pad), pad + row * (height + pad)))
    return sheet


sheet = contact_sheet([image for _bearing, image in frames])
sheet.thumbnail((1100, 1100))
sheet

## Or one long strip

A panorama proper. This needs one measurement first, and skipping it is exactly what makes
a strip come out wrong: a frame is **wider than the angle it was stepped by**. At this
window size the view spans about 48.5 degrees across 640 px while the sweep turns 45
degrees per frame, so butting whole frames together repeats 3.5 degrees at every seam -
around 46 px of duplicated mountain.

So measure what one degree of yaw is worth in pixels, by turning a known amount and
finding how far the picture moved:

In [ ]:
from PIL import ImageChops


def pixels_per_degree(nav, probe_deg=5.0, bearing=0.0):
    """Turns by a known angle and finds the shift it produced."""
    def grey(deg):
        nav.look(bearing_deg=deg, pitch_deg=0)
        nav.wait(settle_ms=400)
        return Image.open(BytesIO(nav.frame("jpg"))).convert("L")

    before, after = grey(bearing), grey(bearing + probe_deg)
    w, h = before.size
    band = (0, h // 3, w, 2 * h // 3)          # the middle third: most detail, least sky
    a, b = before.crop(band), after.crop(band)
    best, best_score = 0, None
    for dx in range(1, w // 3):
        overlap_a = a.crop((dx, 0, w, a.size[1]))
        overlap_b = b.crop((0, 0, w - dx, b.size[1]))
        diff = ImageChops.difference(overlap_a, overlap_b)
        score = sum(count * value for value, count in enumerate(diff.histogram()))
        score /= overlap_a.size[0] * overlap_a.size[1]
        if best_score is None or score < best_score:
            best, best_score = dx, score
    return best / probe_deg


px_per_deg = pixels_per_degree(nav)
print(f"{px_per_deg:.2f} px per degree - the frame spans "
      f"{frames[0][1].size[0] / px_per_deg:.1f} degrees")

Now each frame contributes exactly the slice it stepped by, taken from its middle - where
a rectilinear projection is closest to the cylinder a panorama wants, so the seams advance
by the angle the camera actually turned instead of repeating it.

In [ ]:
slice_px = int(round(360 / STEPS * px_per_deg))
width, height = frames[0][1].size
left = (width - slice_px) // 2                 # the central slice, not the whole frame

strip = Image.new("RGB", (slice_px * len(frames), height))
for index, (_bearing, image) in enumerate(frames):
    strip.paste(image.crop((left, 0, left + slice_px, height)), (index * slice_px, 0))
strip.save("panorama.jpg", quality=90)
print("panorama.jpg:", strip.size, "- a full 360 degrees")
strip.thumbnail((1400, 1400))
strip

It is still not a photographic stitch: the edges of a perspective frame stretch, and only
reprojecting each onto a cylinder removes that. But the content now advances by exactly the
angle the camera turned, which is what "off by some pixels" was.

## A flight

The same loop, moving instead of turning. `move_to` with `await_tiles_ms` is what makes
each frame wait for its own terrain — the reason a scripted flight looks steady rather
than flickering.

In [ ]:
route = [(45.9763, 7.6586), (45.9500, 7.5500), (45.9200, 7.4200), (45.8800, 7.2500)]

flight = []
started = time.time()
for lat, lon in tqdm(route, desc="flight", unit="waypoint"):
    # Each waypoint streams a whole new view, so this is the slow cell: expect tens of
    # seconds per point the first time over a region, and longer if the terrain has never
    # been downloaded. Bounded so it cannot sit for ever.
    nav.move_to(lat, lon, await_tiles_ms=45_000)
    nav.look(bearing_deg=250, pitch_deg=-6)
    nav.set_altitude_asl(4200)
    nav.wait(tiles_timeout_ms=20_000, settle_ms=300)
    flight.append(Image.open(BytesIO(nav.frame("jpg"))))

sheet = contact_sheet(flight, columns=2)
sheet.thumbnail((1100, 1100))
sheet

In [ ]:
nav.close()